# Notebook 06 (Optional): TARNet Deep Uplift Extension

This is an advanced extension notebook. It demonstrates representation-learning causal modeling and is not required to be the production winner.

TARNet intuition:
- shared representation trunk learns common structure
- two heads model treated and control outcomes
- uplift is `y_hat_treated - y_hat_control`

In [ ]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='talk')

if Path.cwd().name == 'notebooks':
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import set_seed, ensure_output_dirs, save_plot
from src.preprocessing import load_hillstrom_data, basic_cleaning, split_features_outcomes_treatment, create_train_valid_test_split
from src.meta_learners import XLearner
from src.evaluation import qini_curve, qini_coefficient, auuc_score

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

SEED = 42
set_seed(SEED, include_torch=True)
OUT_DIRS = ensure_output_dirs(PROJECT_ROOT)
DATA_PATH = PROJECT_ROOT / 'data' / 'hillstrom.csv'

device = torch.device('mps' if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() else 'cpu')
print('Using torch device:', device)

In [ ]:
df = basic_cleaning(load_hillstrom_data(str(DATA_PATH)))
X, y, t = split_features_outcomes_treatment(df, outcome_col='conversion', treatment_col='treatment')
splits = create_train_valid_test_split(X, y, t, test_size=0.2, valid_size=0.2, random_state=SEED, stratify=True)

scaler = StandardScaler()
X_train = scaler.fit_transform(splits['X_train'])
X_valid = scaler.transform(splits['X_valid'])
X_test = scaler.transform(splits['X_test'])

y_train = np.asarray(splits['y_train'], dtype=np.float32)
y_valid = np.asarray(splits['y_valid'], dtype=np.float32)
y_test = np.asarray(splits['y_test'], dtype=np.float32)
t_train = np.asarray(splits['t_train'], dtype=np.float32)
t_valid = np.asarray(splits['t_valid'], dtype=np.float32)
t_test = np.asarray(splits['t_test'], dtype=np.float32)

## TARNet architecture and training loop

In [ ]:
class TARNet(nn.Module):
    """Simple TARNet with shared representation and two heads."""
    def __init__(self, input_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.head_t = nn.Sequential(nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Linear(64, 1))
        self.head_c = nn.Sequential(nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, x):
        z = self.shared(x)
        y1 = self.head_t(z).squeeze(-1)
        y0 = self.head_c(z).squeeze(-1)
        return y0, y1, z

model = TARNet(input_dim=X_train.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.BCEWithLogitsLoss()

train_ds = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32),
    torch.tensor(t_train, dtype=torch.float32),
)
valid_ds = TensorDataset(
    torch.tensor(X_valid, dtype=torch.float32),
    torch.tensor(y_valid, dtype=torch.float32),
    torch.tensor(t_valid, dtype=torch.float32),
)

train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=1024, shuffle=False)

In [ ]:
best_val = float('inf')
best_state = None
patience = 5
stale = 0
history = []

for epoch in range(30):
    model.train()
    train_losses = []
    for xb, yb, tb in train_loader:
        xb, yb, tb = xb.to(device), yb.to(device), tb.to(device)
        optimizer.zero_grad()
        y0_logit, y1_logit, _ = model(xb)
        factual_logit = tb * y1_logit + (1 - tb) * y0_logit
        loss = criterion(factual_logit, yb)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    model.eval()
    valid_losses = []
    with torch.no_grad():
        for xb, yb, tb in valid_loader:
            xb, yb, tb = xb.to(device), yb.to(device), tb.to(device)
            y0_logit, y1_logit, _ = model(xb)
            factual_logit = tb * y1_logit + (1 - tb) * y0_logit
            vloss = criterion(factual_logit, yb)
            valid_losses.append(vloss.item())

    tr = float(np.mean(train_losses))
    va = float(np.mean(valid_losses))
    history.append({'epoch': epoch + 1, 'train_loss': tr, 'valid_loss': va})
    print(f'Epoch {epoch+1:02d} | train={tr:.4f} valid={va:.4f}')

    if va < best_val:
        best_val = va
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        stale = 0
    else:
        stale += 1
        if stale >= patience:
            print('Early stopping triggered.')
            break

if best_state is not None:
    model.load_state_dict(best_state)
torch.save(model.state_dict(), OUT_DIRS['models'] / 'nb06_tarnet_best.pt')
loss_df = pd.DataFrame(history)
loss_df.to_csv(OUT_DIRS['tables'] / 'nb06_tarnet_training_history.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(loss_df['epoch'], loss_df['train_loss'], label='train')
ax.plot(loss_df['epoch'], loss_df['valid_loss'], label='valid')
ax.set_title('TARNet Training Curve')
ax.set_xlabel('Epoch')
ax.set_ylabel('BCE loss')
ax.legend()
plt.tight_layout()
save_plot(fig, OUT_DIRS['figures'] / 'nb06_tarnet_training_curve.png')
plt.show()

## ITE estimation and comparison

In [ ]:
model.eval()
with torch.no_grad():
    x_tensor = torch.tensor(X_test, dtype=torch.float32, device=device)
    y0_logit, y1_logit, z = model(x_tensor)
    y0_prob = torch.sigmoid(y0_logit).cpu().numpy()
    y1_prob = torch.sigmoid(y1_logit).cpu().numpy()
    tarnet_uplift = y1_prob - y0_prob
    rep = z.cpu().numpy()

x_learner = XLearner(random_state=SEED).fit(splits['X_train'], splits['t_train'], splits['y_train'])
x_uplift = x_learner.predict_uplift(splits['X_test'])

metrics = pd.DataFrame([
    {'model': 'tarnet', 'qini': qini_coefficient(y_test, t_test, tarnet_uplift), 'auuc': auuc_score(y_test, t_test, tarnet_uplift)},
    {'model': 'x_learner', 'qini': qini_coefficient(y_test, t_test, x_uplift), 'auuc': auuc_score(y_test, t_test, x_uplift)},
]).sort_values('qini', ascending=False).reset_index(drop=True)
display(metrics)
metrics.to_csv(OUT_DIRS['tables'] / 'nb06_tarnet_vs_xlearner_metrics.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for name, score in [('tarnet', tarnet_uplift), ('x_learner', x_uplift)]:
    c = qini_curve(y_test, t_test, score)
    ax.plot(c['fraction_targeted'], c['incremental_outcomes'], label=name)
ax.set_title('Qini: TARNet vs X-Learner')
ax.set_xlabel('Fraction targeted')
ax.set_ylabel('Cumulative incremental conversions')
ax.legend()
plt.tight_layout()
save_plot(fig, OUT_DIRS['figures'] / 'nb06_tarnet_qini_comparison.png')
plt.show()

## Representation visualization (UMAP or fallback)

In [ ]:
try:
    import umap
    reducer = umap.UMAP(random_state=SEED)
    emb = reducer.fit_transform(rep)
    emb_name = 'UMAP'
except Exception:
    from sklearn.decomposition import PCA
    emb = PCA(n_components=2, random_state=SEED).fit_transform(rep)
    emb_name = 'PCA (fallback)'

viz = pd.DataFrame({'x': emb[:, 0], 'y': emb[:, 1], 'treatment': t_test, 'conversion': y_test})
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.scatterplot(data=viz.sample(min(5000, len(viz)), random_state=SEED), x='x', y='y', hue='treatment', alpha=0.6, ax=axes[0])
axes[0].set_title(f'{emb_name} of shared representation (colored by treatment)')
sns.scatterplot(data=viz.sample(min(5000, len(viz)), random_state=SEED), x='x', y='y', hue='conversion', alpha=0.6, ax=axes[1])
axes[1].set_title(f'{emb_name} of shared representation (colored by conversion)')
plt.tight_layout()
save_plot(fig, OUT_DIRS['figures'] / 'nb06_tarnet_umap.png')
plt.show()

## Discussion
- TARNet can help when complex response surfaces benefit from shared representation learning.
- On medium-sized tabular marketing datasets, tree-based methods may remain competitive or superior.
- Deep causal models are valuable for experimentation depth but are not guaranteed business winners.

## End-of-notebook summary
This optional notebook demonstrates advanced modeling depth with a causal neural architecture, while keeping evaluation policy-focused and comparable to classical approaches.